# 07 — Live paper trading dashboard

**Default mode:** in-memory paper broker (no broker account needed).

**Optional:** Alpaca paper broker (set `USE_ALPACA = True` below; needs ALPACA_API_KEY).

Re-run the 'Tick' cell to: pull latest price → recompute signal → check risk → place
orders → snapshot portfolio.

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import uuid, time
from src.config import SAFETY, KEYS
from src.data.yfinance_client import YFinanceClient
from src.strategies.spy_vol_target import SPYVolTargetStrategy, VolTargetConfig
from src.live.paper_broker import PaperBroker
from src.live.risk_monitor import RiskMonitor, RiskState
from src.live.broker_base import Order
from src.backtest.engine import RiskLimits
from src.backtest.costs import CostModel

USE_ALPACA = False    # flip to True if you have Alpaca paper creds in .env
STARTING_CASH = 100_000
run_id = uuid.uuid4().hex[:12]
print('run_id:', run_id, ' paper mode (live disabled by default):', not SAFETY.can_trade_live)

In [ ]:
yf = YFinanceClient()
def latest_price(symbol):
    df = yf.get_daily_bars(symbol, start='2024-01-01', use_cache=False)
    return float(df['adj_close'].iloc[-1])

if USE_ALPACA and KEYS.alpaca_id:
    from src.live.alpaca_broker import AlpacaBroker
    broker = AlpacaBroker(live=False)
else:
    broker = PaperBroker(run_id=run_id, starting_cash=STARTING_CASH,
                          price_fn=latest_price, costs=CostModel())
broker

In [ ]:
limits = RiskLimits(max_gross_leverage=1.0, max_single_asset_weight=0.5,
                    drawdown_warn=-0.03, drawdown_derisk=-0.05, drawdown_stop=-0.08)
state = RiskState(run_id=run_id, limits=limits, starting_equity=STARTING_CASH)
monitor = RiskMonitor(broker, state)
print('monitor armed')

### Tick — run signal → rebalance to target

This pulls the latest SPY price, computes target weight from the SPY vol-target strategy, and submits the required order through the risk monitor.

In [ ]:
def tick():
    df = yf.get_daily_bars('SPY', start='2023-01-01', use_cache=False)
    prices = pd.DataFrame({'SPY': df['adj_close']})
    sig = SPYVolTargetStrategy(VolTargetConfig(symbol='SPY', target_vol=0.10, rv_window=21))\
          .signal(prices)
    target_w = float(sig['SPY'].iloc[-1])
    acct = broker.account()
    equity = acct.get('equity', STARTING_CASH)
    px = broker.last_price('SPY')
    target_qty = (target_w * equity) / px
    cur_qty = next((p.qty for p in broker.positions() if p.symbol == 'SPY'), 0.0)
    delta = target_qty - cur_qty
    monitor.update_equity(equity)
    if abs(delta * px) >= 50:
        side = 'buy' if delta > 0 else 'sell'
        result = monitor.submit(Order(symbol='SPY', side=side, qty=round(abs(delta), 4),
                                       reason='spy_vt tick'))
    else:
        result = dict(status='no_op')
    return dict(target_w=target_w, target_qty=round(target_qty,4),
                 cur_qty=cur_qty, equity=equity, px=px, result=result)

tick()

### Dashboard

In [ ]:
acct = broker.account()
from src.data.storage import query
fills = query('SELECT * FROM fills WHERE run_id=? ORDER BY ts DESC', (run_id,))
events = query('SELECT * FROM risk_events WHERE run_id=? ORDER BY ts DESC', (run_id,))
snapshots = query('SELECT * FROM portfolio_snapshots WHERE run_id=? ORDER BY ts DESC LIMIT 30', (run_id,))

print('=== Account ===')
for k, v in acct.items(): print(f'  {k}: {v}')
print('\n=== Positions ===')
for p in broker.positions(): print(f'  {p.symbol}: qty={p.qty:.4f} avg=${p.avg_px:.2f}')
print('\n=== Recent fills (last 5) ===')
fills.head() if not fills.empty else 'none'

In [ ]:
events.head() if not events.empty else 'no risk events'

### Optional: loop

Uncomment to run a tick every N seconds for a duration. **Note:** yfinance daily bars don't change intraday, so for true live ticking you want intraday Polygon/Alpaca.

In [ ]:
# from src.live.scheduler import loop_until
# loop_until(tick, every_seconds=60, until=pd.Timestamp.utcnow() + pd.Timedelta(hours=1))